# Embedding & Retrieval — walkthrough

Corpus is built (`data/corpus.json`, 4,000 docs). This notebook covers the next two steps and
explains **exactly what is stored** and **exactly what happens at query time**.

```
corpus.json ──embed──►  vectors_single.npy   (3,879 × 384)
                    └─►  vectors_multi.npy    (  121 × 384)

query ──► search_single()  cosine + BM25 → RRF → top 5
      └─► search_multi()   cosine over 121 → top 1
```

The notebook **drives `embed.py` and `search.py`** — it does not reimplement them. Anything you
change here should be changed in those files, so the Streamlit app gets it too.

**The one-line mental model:** every doc becomes a unit vector; a query becomes a unit vector;
similarity is a dot product; 4,000 dot products is one matrix multiply, which takes ~1 ms.

## 0. Setup

Run once if the imports below fail. `torch` comes from the CPU-only index — the default wheel is
a 2.5 GB CUDA build we have no use for.

In [1]:
# %pip install numpy rank-bm25 python-dotenv
# %pip install torch --index-url https://download.pytorch.org/whl/cpu
# %pip install sentence-transformers

In [2]:
import os
import sys
import time
from collections import Counter
from pathlib import Path

import numpy as np

# The project modules import as `data.raw.config`, so the project ROOT must be on sys.path.
ROOT = Path.cwd()
while not (ROOT / "data" / "corpus.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from data.raw.config import (
    ABSTAIN_THRESHOLD,
    CORPUS,
    EMBED_DIMS,
    EMBED_MODEL,
    FIQH_THRESHOLD,
    LOCAL_EMBED_MODEL,
    RRF_K,
    TOP_K_ISLAMQA,
    VECTORS_MULTI,
    VECTORS_SINGLE,
)
from data.raw.schema import load_corpus

USE_LOCAL = not os.getenv("OPENAI_API_KEY")

print(f"root     {ROOT}")
print(f"corpus   {CORPUS.relative_to(ROOT)}  exists={CORPUS.exists()}")
print(f"backend  {LOCAL_EMBED_MODEL} (local, 384d)" if USE_LOCAL
      else f"backend  {EMBED_MODEL} ({EMBED_DIMS}d, OpenAI)")

root     C:\Users\DELL\ummah_project
corpus   data\corpus.json  exists=True
backend  BAAI/bge-small-en-v1.5 (local, 384d)


## 1. What is a "chunk" here?

**One `Doc` = one chunk.** There is no sub-document splitting, and that is a deliberate choice, not
a shortcut:

| | Generic RAG | Here |
|---|---|---|
| Unit | 500-token slice of a long document | one complete fatwa (Q + A) |
| Why | source docs are books/PDFs covering many topics | a fatwa is *already* the atomic unit — one question, one ruling |
| Risk of splitting | — | half a ruling retrieved without its conditions → **you misattribute a position to a scholar** |

Splitting an answer mid-way is actively dangerous in this domain: the caveat *"…unless the
transaction involves a deferred payment"* often lives three paragraphs below the verdict. Retrieve
the verdict without the caveat and the card is wrong.

Each Doc exposes two derived texts (`schema.py`), and they are **not** the same:

- **`embed_text` = `title + question`** — this is what gets vectorised. Queries *are* questions, and
  question↔question similarity is much stronger than question↔answer. Embedding the 3,000-char
  answer would dilute the vector toward the generic Islamic vocabulary every fatwa shares.
- **`bm25_text` = `title + question + answer + positions`** — the full substance. The answer is
  still fully searchable, just through the keyword half of retrieval rather than the vector half.

Neither is stored in `corpus.json` — both are `@property`, derived on load. That is why the JSON is
15 MB rather than 30 MB.

In [3]:
docs = load_corpus(CORPUS)
single = [d for d in docs if not d.is_multi_school]   # answer -> one card
multi = [d for d in docs if d.is_multi_school]        # positions -> four cards

print(f"{len(docs)} docs = {len(single)} single_source + {len(multi)} multi_school\n")

for src, n in Counter(d.source for d in docs).most_common():
    print(f"  {src:<12} {n:>5}")
print()
for o, n in Counter(d.orientation for d in docs).most_common():
    print(f"  {o:<20} {n:>5}")

4000 docs = 3879 single_source + 121 multi_school

  islamqa       1589
  islamqaorg    1287
  askimam       1003
  fiqhqa         121

  Salafi                1589
  Hanafi                1418
  Shafi'i                415
  Maliki                 275
  Hanbali                182
  Four Sunni schools     121


In [4]:
# The two texts side by side. This is the single most important thing to
# understand before reading the retrieval code.
d = single[0]
print(f"id       {d.id}")
print(f"source   {d.source_label}  ({d.orientation})\n")
print(f"--- embed_text  ({len(d.embed_text):>5} chars) -> THIS becomes the vector")
print(d.embed_text[:400])
print(f"\n--- bm25_text   ({len(d.bm25_text):>5} chars) -> THIS is keyword-indexed")
print(d.bm25_text[:400] + " ...")

lens = np.array([len(x.embed_text) for x in docs])
print(f"\nembed_text length: median {int(np.median(lens))}, p95 {int(np.percentile(lens, 95))}, "
      f"max {lens.max()} chars")
print("(bge-small truncates at 512 tokens ~ 2,000 chars - check p95 sits under that)")

id       islamqa:145290
source   IslamQA.info  (Salafi)

--- embed_text  (  540 chars) -> THIS becomes the vector
She stopped praying when she was sick, then she died; do her heirs have to do anything?
My mother died, and she owed two months’ prayers because of cancer; she was intending to make them up. She also owes the fast of the Ramadan before the last, when she was healthy. My question is: what is the correct action with regard to her acts of worship? Please note that I have sisters; can we cooperate to 

--- bm25_text   ( 6146 chars) -> THIS is keyword-indexed
She stopped praying when she was sick, then she died; do her heirs have to do anything?
My mother died, and she owed two months’ prayers because of cancer; she was intending to make them up. She also owes the fast of the Ramadan before the last, when she was healthy. My question is: what is the correct action with regard to her acts of worship? Please note that I have sisters; can we cooperate to  ...

embed_text length: me

## 2. Storage — what actually goes on disk

Two `.npy` files, and **no database, no vector store, no ids inside the file**. Three decisions
make that work:

**a) Two indexes, not one.** The 121 multi-school records would essentially never win top-k against
3,879 single-source fatwas, so the four-school comparison panel — the headline feature — would
silently never fire. They get their own index. Searching 121 vectors costs ~0 ms, so the panel
fires whenever there is a real match, independent of what the main index returns.

**b) Row order is the primary key.** Row `i` of `vectors_single.npy` is `single[i]`, where `single`
is `corpus.json` filtered by `record_type` in file order. No id column, no sidecar mapping — the
contract is reproduced by `embed.py` and `search.py` running the same filter. `Index.__init__`
asserts the row counts match and refuses to load if they don't, so a stale `.npy` fails loudly at
startup instead of returning confidently wrong documents.

**c) L2-normalised float32 on write.** Once every row has ‖v‖ = 1,

$$\cos(q, d) = \frac{q \cdot d}{\|q\|\,\|d\|} = q \cdot d$$

so cosine similarity over the whole corpus is `V @ q` — **one matrix multiply, no per-row loop, no
normalisation at query time**. That is the "faster retrieval" the plan points at. 4,000 × 384
float32 is 6 MB; at this scale a brute-force matmul beats an approximate index (FAISS/HNSW) on both
latency and recall, and costs zero build time.

### 2.1 Run the embedder

First run downloads `bge-small-en-v1.5` (~130 MB) and takes a few minutes on CPU. It writes both
`.npy` files; re-running is idempotent. Equivalent to `python embed.py --local` from the shell —
we call `embed.py`'s own functions so the notebook and the app can never drift apart.

In [5]:
import embed  # noqa: E402  -- embed.py lives at the project root

FORCE_REEMBED = False   # flip to True after changing embed_text or the corpus

backend = embed.embed_local if USE_LOCAL else embed.embed_openai

for name, group, path in (("single_source", single, VECTORS_SINGLE),
                          ("multi_school", multi, VECTORS_MULTI)):
    if path.exists() and not FORCE_REEMBED:
        print(f"{name:<14} cached  {np.load(path).shape}  ({path.name})")
        continue
    print(f"{name:<14} embedding {len(group)} docs ...")
    t0 = time.perf_counter()
    vecs = backend([g.embed_text for g in group])       # <- L2-normalised inside embed.py
    np.save(path, vecs)
    print(f"{name:<14} -> {path.name}  {vecs.shape}  {vecs.nbytes / 1e6:.1f} MB  "
          f"in {time.perf_counter() - t0:.1f}s")

single_source  cached  (3879, 384)  (vectors_single.npy)
multi_school   cached  (121, 384)  (vectors_multi.npy)


In [6]:
# Verify the three storage invariants before trusting any search result.
V_single = np.load(VECTORS_SINGLE)
V_multi = np.load(VECTORS_MULTI)

print(f"vectors_single  {V_single.shape}  {V_single.dtype}  {V_single.nbytes / 1e6:.1f} MB")
print(f"vectors_multi   {V_multi.shape}  {V_multi.dtype}  {V_multi.nbytes / 1e6:.1f} MB\n")

assert len(V_single) == len(single), "row-count drift - re-run the embed cell"
assert len(V_multi) == len(multi), "row-count drift - re-run the embed cell"
print(f"[ok] row order   V_single[i] <-> single[i]   ({len(single)} rows)")

norms = np.linalg.norm(V_single, axis=1)
assert np.allclose(norms, 1.0, atol=1e-4), norms[:5]
print(f"[ok] unit norm   min {norms.min():.6f}  max {norms.max():.6f}  -> dot product == cosine")

assert V_single.dtype == np.float32
print("[ok] float32     half the RAM of float64, no measurable accuracy loss at 384d")

vectors_single  (3879, 384)  float32  6.0 MB
vectors_multi   (121, 384)  float32  0.2 MB

[ok] row order   V_single[i] <-> single[i]   (3879 rows)
[ok] unit norm   min 1.000000  max 1.000000  -> dot product == cosine
[ok] float32     half the RAM of float64, no measurable accuracy loss at 384d


## 3. Retrieval — what happens on a query

`search.py` runs **two searches per query**, and fuses two signals inside the first one.

**Vector half** — `cos = V_single @ q`. Catches paraphrase: *"is a home loan halal"* matches a
fatwa titled *"Ruling on interest-based mortgages"* with no shared keyword.

**Keyword half** — BM25 over `bm25_text`. Catches the rare domain terms embeddings are worst at:
*riba, masah, mudarabah, istihada, khul', talaq*. A general-purpose embedder has seen these few
times; BM25 doesn't care what a word means, only that it is rare and present.

**Fusion — RRF, not a weighted sum.** Cosine lives on roughly [0, 1]; BM25 is unbounded and
corpus-dependent. Adding them requires a calibration constant that drifts every time the corpus
changes. Reciprocal rank fusion throws the scores away and keeps only the **ranks**:

$$\text{RRF}(d) = \sum_{r \in \{\text{vec},\, \text{bm25}\}} \frac{1}{K + \text{rank}_r(d)}, \qquad K = 60$$

A doc ranked #1 by both scores 2/61. Ranked #1 by one and #500 by the other, ~1/61. So agreement
between two independent signals is what wins — and there is nothing to tune.

**Diversity cap.** `MAX_PER_SOURCE = 2`: no single darul-ifta may take more than 2 of the 5 slots.
Without it one prolific site fills the page and the multi-orientation corpus is wasted.

**Cost.** One embed call for the query (the only slow step, ~20 ms local), then a matmul over
3,879 rows and a BM25 scan. Everything after the query embedding is single-digit milliseconds.

In [7]:
from search import Index  # noqa: E402

t0 = time.perf_counter()
idx = Index()   # loads corpus + both .npy + builds the BM25 index. Do this ONCE.
print(f"index built in {time.perf_counter() - t0:.1f}s  "
      f"({len(idx.single)} single, {len(idx.multi)} multi)")
print("in the app this sits behind @st.cache_resource so it happens once per process")

index built in 0.9s  (3879 single, 121 multi)
in the app this sits behind @st.cache_resource so it happens once per process


### 3.1 Sample query → top-k chunks

Change `QUERY` and re-run. The `score` shown is **cosine** — the interpretable number — even though
the *ordering* is RRF.

In [8]:
QUERY = "Is a conventional mortgage permissible?"
K = 5

# Index() loads the vectors but NOT the embedding model - that happens lazily on
# the first embed_query(). Warm it here so the timing below measures retrieval
# rather than a one-off ~15s model load. app.py pays that same cost exactly once,
# inside @st.cache_resource.
idx.embed_query("warmup")

t0 = time.perf_counter()
hits = idx.search_single(QUERY, k=K)
print(f'"{QUERY}"   -> {len(hits)} hits in {(time.perf_counter() - t0) * 1000:.0f} ms')
print()

for rank, h in enumerate(hits, 1):
    print(f"{rank}. [cos {h.score:.3f}]  {h.title}")
    print(f"   {h.source_label}  |  {h.orientation}  |  {', '.join(h.categories[:3]) or '-'}")
    print(f"   {h.url}")
    print(f"   {' '.join(h.answer.split())[:220]} ...")
    print()


C:\Users\DELL\ummah_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8290.70it/s]

"Is a conventional mortgage permissible?"   -> 5 hits in 25 ms

1. [cos 0.784]  Are Mortgages Haram?
   IslamQA.info  |  Salafi  |  Interest
   https://islamqa.info/en/answers/159213/are-mortgages-haram
   What is a mortgage? A mortgage is a haram riba-based transaction that is based on a loan with interest in which the owner of the money takes as collateral the property for the purchase of which the borrower is taking out ...

2. [cos 0.776]  Should he mortgage his house to a non-Islamic bank so that he can buy another house?
   IslamQA.info  |  Salafi  |  Borrowing lending, Interest
   https://islamqa.info/en/answers/83321/should-he-mortgage-his-house-to-a-non-islamic-bank-so-that-he-can-buy-another-house
   If what is meant is a loan from the bank, and mortgaging the house to secure the debt, then it depends: Firstly: if the loan is to be repaid with something extra (interest), then it is a riba-based loan which is haraam.  ...

3. [cos 0.710]  Do we receive interests from conventio

In [9]:
# Second search, same query: the 121-row multi-school index. Always runs, always cheap.
panel = idx.search_multi(QUERY, k=1)
if panel:
    p = panel[0]
    fires = p.score >= FIQH_THRESHOLD
    print(f"best multi-school match  cos {p.score:.3f}  (threshold {FIQH_THRESHOLD})")
    print(f"  {p.title}")
    print(f"  panel fires: {fires}"
          + ("" if fires else "  -> UI shows a 'single-source' badge, not a silent gap"))
    if fires:
        for pos in p.stated_positions:
            print(f"\n  {pos.school_label:<9} {' '.join(pos.text.split())[:150]}")

best multi-school match  cos 0.599  (threshold 0.72)
  Marriage Contract by Proxy
  panel fires: False  -> UI shows a 'single-source' badge, not a silent gap


### 3.2 Why hybrid — watch the two halves disagree

This is the cell to look at if you ever wonder whether BM25 is earning its place. Vector-only and
BM25-only top-5 for the same query, next to the fused result.

In [10]:
from search import tokenise  # noqa: E402

q = idx.embed_query(QUERY)
cos = idx.v_single @ q                            # vector half
bm = idx.bm25.get_scores(tokenise(QUERY))         # keyword half
vec_rank, bm_rank = np.argsort(-cos), np.argsort(-bm)

fused = np.zeros(len(idx.single))                 # RRF, exactly as search.py does it
for r, i in enumerate(vec_rank[:200]):
    fused[i] += 1.0 / (RRF_K + r)
for r, i in enumerate(bm_rank[:200]):
    fused[i] += 1.0 / (RRF_K + r)

def col(order, score, label, fmt="{:.3f}"):
    print(label)
    for r, i in enumerate(order[:5], 1):
        print(f"  {r}. {fmt.format(score[i]):>7}  {idx.single[i].title[:62]}")
    print()

col(vec_rank, cos, "VECTOR only  (cosine)")
col(bm_rank, bm, "BM25 only    (keyword)", "{:.1f}")
col(np.argsort(-fused), fused, "RRF FUSED    (what search_single ranks on, pre-diversity-cap)", "{:.4f}")

overlap = len(set(vec_rank[:5].tolist()) & set(bm_rank[:5].tolist()))
print(f"top-5 overlap between the two halves: {overlap}/5 -> "
      + ("the halves mostly agree here" if overlap >= 4
         else "the halves disagree, so fusion is doing real work"))

VECTOR only  (cosine)
  1.   0.784  Are Mortgages Haram?
  2.   0.780  Buying a house through the bank
  3.   0.779  Is it permissible to benefit from items purchased through trad
  4.   0.776  Should he mortgage his house to a non-Islamic bank so that he 
  5.   0.758  Should he take a riba-based loan if he does not intend to pay 

BM25 only    (keyword)
  1.    19.2  Are Mortgages Haram?
  2.    18.5  Do we receive interests from conventional banks or dividends
  3.    18.3  We have friends who work in a bank that is commercial banks ..
  4.    17.7  Ruling on managing property for someone else that was brought 
  5.    17.6  What should a person who
 his father borrowed with interest(Ri

RRF FUSED    (what search_single ranks on, pre-diversity-cap)
  1.  0.0333  Are Mortgages Haram?
  2.  0.0313  Should he mortgage his house to a non-Islamic bank so that he 
  3.  0.0299  Should he go for an interest-based mortgage if that is cheaper
  4.  0.0297  Is it permissible to rent out a mor

## 4. The full decision — `idx.retrieve()`

This is the only method `app.py` needs to call. It returns the three states the UI must render:

| State | Condition | UI |
|---|---|---|
| **abstain** | top score < `ABSTAIN_THRESHOLD` | "I don't have a sourced fatwa on this" |
| **four-school** | multi-school hit ≥ `FIQH_THRESHOLD` | comparison panel + coverage badge |
| **single-source** | otherwise | ruling cards + orientation badge |

Coverage is a **visible state, never a silent absence** — the panel not rendering must read as a
stated fact, not a bug.

Below: all 7 demo queries. The target behaviour is that the first three fire the panel, the middle
three do not, and Mars abstains.

> ⚠️ **With the shipped thresholds (0.45 / 0.30) this test currently fails** — every query fires the
> panel and nothing abstains. Nothing is broken in retrieval; the *ranking* above is good. The
> thresholds were chosen for `text-embedding-3-small`, and `bge-small` puts cosines on a completely
> different scale: it scores even unrelated pairs around 0.6–0.7, so a 0.45 cut-off is below the
> noise floor. **Thresholds are a property of the embedding model, not of the corpus** — swapping
> the backend invalidates them. §4.1 measures the right values.

In [11]:
from data.raw.demo_queries import ALL  # noqa: E402

for spec in ALL:
    r = idx.retrieve(spec["query"])
    state = ("ABSTAIN" if r["abstain"]
             else "four-school" if r["show_schools"] else "single-source")
    print(f"{state:<14} top={r['top_score']:.3f}   {spec['query']}")
    if r["panel"]:
        print(f"               PANEL {r['panel'].score:.3f}  {r['panel'].title[:58]}")
    for h in r["hits"][:3]:
        print(f"               {h.score:.3f}  {h.orientation:<9} {h.title[:52]}")
    want = spec.get("expect_doc")
    if want:
        got = [h.id for h in r["hits"]] + ([r["panel"].id] if r["panel"] else [])
        print(f"               expect {want}: {'FOUND' if want in got else 'MISSING'}")
    print()

four-school    top=0.847   Does touching a woman break wudu?
               PANEL 0.797  The Nullification of Wudu by Touching a Woman
               0.845  Hanafi    Will touching the private part break wudhu?
               0.811  Shafi'i   Touching hair of a marriageable woman
               0.733  Salafi    Does Kissing Break Wudu?
               expect fiqhqa:21: FOUND

four-school    top=0.843   Where do you place your hands in prayer?
               PANEL 0.738  Qabd (placing the right hand over the left) in prayer.
               0.843  Hanbali   Where should I put my hands in prayer? (Hanbali Fiqh
               0.711  Hanbali   In the janaza prayer, do I raise my hands with every
               0.701  Salafi    Can You Pray in a Moving Car?
               expect fiqhqa:1: FOUND

four-school    top=0.905   Does a divorce pronounced during menstruation count?
               PANEL 0.905  Divorce During Menstruation
               0.789  Shafi'i   Divorce and the Waiting Period
 

single-source  top=0.784   Is a conventional mortgage permissible?
               0.784  Salafi    Are Mortgages Haram?
               0.776  Salafi    Should he mortgage his house to a non-Islamic bank s
               0.710  Hanafi    Do we receive interests from conventional banks or d



single-source  top=0.853   What is the ruling on buying and selling bitcoin?
               0.853  Salafi    What is the ruling on buying and selling bitcoin?
               0.734  Hanafi    What are the basic rules of selling and buying accor
               0.727  Hanafi    Playing with and selling Pokémon cards
               expect islamqa:360668: FOUND

single-source  top=0.858   Is life insurance allowed in Islam?
               0.814  Salafi    Can Muslims Benefit from Company-Provided Health or 
               0.858  Salafi    Is Life Insurance Halal in Islam?
               0.782  Hanafi    I had Life insurance policy here in INDIA since 4 ye



ABSTAIN        top=0.710   What are the rules for prayer on Mars?
               0.684  Hanafi    Salah Whilst Travelling – Can we Combine Prayers?
               0.680  Hanafi    What are the conditions for praying Istikhara Salah?
               0.666  Maliki    Should We Wear Special Clothes for Prayer (Maliki)?



### 4.1 Calibrating the two thresholds

A threshold is just a **separating line between two measured populations**, so measure them rather
than guessing:

- `FIQH_THRESHOLD` — panel scores where the four-school panel *should* fire, vs. where it should not.
- `ABSTAIN_THRESHOLD` — top scores on answerable queries, vs. on the unanswerable one.

The cell below prints both populations and the window that separates them. Pick the **midpoint** of
each window and put it in `data/raw/config.py`.

Two caveats worth saying out loud to a judge:

1. **7 queries is a tiny calibration set.** The Mars query is the only negative example, so
   `ABSTAIN_THRESHOLD` rests on a single data point. Add 3–4 more off-corpus queries ("ruling on
   quantum computing", "how do I fix my car") before trusting it.
2. **The margin is narrow.** Mars scores higher against the 121-row index than one genuine
   four-school query does — because "prayer on Mars" *is* about prayer, so it legitimately looks
   close to a prayer record. The separating line exists, but it is thin, which is exactly the kind
   of thing to know before a demo rather than during one.

In [12]:
from data.raw.demo_queries import ABSTAIN, FOUR_SCHOOL, SINGLE_SOURCE  # noqa: E402

rows = []
for group, specs in (('four', FOUR_SCHOOL), ('single', SINGLE_SOURCE), ('abstain', ABSTAIN)):
    for s in specs:
        hits = idx.search_single(s['query'])
        best = idx.search_multi(s['query'], k=1)[0]
        rows.append((group, best.score, max([h.score for h in hits] + [best.score]), s['query']))

print(f"{'group':<8} {'panel':>6} {'top':>6}   query")
for g, panel_score, top, query in rows:
    print(f"{g:<8} {panel_score:>6.3f} {top:>6.3f}   {query[:52]}")

# FIQH_THRESHOLD separates 'panel should fire' from 'panel should stay silent'.
fire = [r[1] for r in rows if r[0] == 'four']
quiet = [r[1] for r in rows if r[0] != 'four']
# ABSTAIN_THRESHOLD separates answerable from unanswerable.
answerable = [r[2] for r in rows if r[0] != 'abstain']
unanswerable = [r[2] for r in rows if r[0] == 'abstain']

for name, lo, hi, cur in (('FIQH_THRESHOLD', max(quiet), min(fire), FIQH_THRESHOLD),
                          ('ABSTAIN_THRESHOLD', max(unanswerable), min(answerable),
                           ABSTAIN_THRESHOLD)):
    ok = lo < hi
    print()
    print(name)
    print(f"  must be  > {lo:.3f}  and  <= {hi:.3f}"
          + (f"   -> set {(lo + hi) / 2:.2f}   (margin {hi - lo:.3f})" if ok
             else "   -> NO SEPARATING VALUE EXISTS on this query set"))
    print(f"  current  {cur}  -> "
          + ('OK' if lo < cur <= hi else 'MISCALIBRATED for this backend'))


group     panel    top   query
four      0.797  0.847   Does touching a woman break wudu?
four      0.738  0.843   Where do you place your hands in prayer?
four      0.905  0.905   Does a divorce pronounced during menstruation count?
single    0.599  0.784   Is a conventional mortgage permissible?
single    0.616  0.853   What is the ruling on buying and selling bitcoin?
single    0.649  0.858   Is life insurance allowed in Islam?
abstain   0.710  0.710   What are the rules for prayer on Mars?

FIQH_THRESHOLD
  must be  > 0.710  and  <= 0.738   -> set 0.72   (margin 0.028)
  current  0.72  -> OK

ABSTAIN_THRESHOLD
  must be  > 0.710  and  <= 0.784   -> set 0.75   (margin 0.074)
  current  0.75  -> OK


## 5. Recap

**Stored:** `corpus.json` (content + provenance only) plus two L2-normalised float32 `.npy` files.
Row order *is* the join key; `embed_text`/`bm25_text` are derived on load, never persisted.

**Retrieved:** query → one vector → `V @ q` matmul + BM25 → RRF fusion → per-source diversity cap →
top-k. Plus an independent cosine search over the 121-row multi-school index, so the four-school
panel can fire regardless of what the main index returns.

**Re-run the embed cell whenever** the corpus changes or `embed_text` changes. `Index.__init__`
refuses to load mismatched vectors, so you will be told — but only at startup.

**Next:** `generate.py` turns these hits into `RulingCard`s — one card per source, never merged.

## 6. Generation - from retrieved chunks to ruling cards

Retrieval hands us documents. Generation has to turn them into something readable **without
inventing a ruling**, which is the whole difficulty: the natural thing for an LLM to do with five
fatwas is average them into one confident answer, and that answer is a position no scholar holds.

So this layer is deliberately narrow. It **extracts and attributes**. It never adjudicates.

```
idx.retrieve(query)
        |
        +-- abstain?        -> stop. 0 LLM calls.
        |
        +-- panel Doc       -> 4 RulingCards, text copied VERBATIM from positions
        |                      + 1 batched call for the verdict enums          1 call
        |
        +-- top-k hits      -> 1 structured call each, scoped to ONE document  <=3 calls
        |
        +-- all cards       -> 1 call -> Comparison (no overall verdict)       1 call
```

Three structural guards, each of which is a piece of code rather than a line in the prompt:

| Guard | How it is enforced |
|---|---|
| No merged verdict | `Comparison` has **no verdict field**. There is nowhere to put one. |
| No mis-attribution | `doc_id` and `attribution` come from the `Doc`, never from the model. |
| No invented citations | every `evidences[].quote` is substring-checked against its source doc. |

The model only ever fills the fields it is genuinely qualified to fill.

In [13]:
import generate  # noqa: E402
from generate import (  # noqa: E402
    answer, compare, disambiguate, render, sampling_args, school_cards,
    source_card, verify_quotes,
)
from data.raw.config import LLM_MAX_TOKENS, LLM_MODEL  # noqa: E402

# generate.py calls load_dotenv() on import, so the key is already in the env if
# .env exists. Everything below is skipped without one, so the notebook still
# runs end-to-end as a retrieval walkthrough on a machine with no credentials.
RUN_LLM = bool(os.getenv("ANTHROPIC_API_KEY"))

print(f"model       {LLM_MODEL}")
print(f"max_tokens  {LLM_MAX_TOKENS}")
print(f"sampling    {sampling_args(LLM_MODEL)}")
print(f"RUN_LLM     {RUN_LLM}")

model       claude-sonnet-5
max_tokens  16000
sampling    {'temperature': 1.0}
RUN_LLM     True


### 6.1 Temperature, and why the code sometimes omits it

We want **temperature 1** here. That is not a typo and it is not carelessness:

- The model is **reading a document we supply**, not recalling anything. Grounding comes from the
  retrieved context and from structured outputs, not from a sampling parameter.
- The output is **schema-constrained**. `verdict` can only ever be one of six enum values no matter
  how the sampler behaves; there is no distribution over which a lower temperature would help.
- Where it does matter - the `reasoning` and `turns_on` prose - temperature 0 produces flatter,
  more templated text without being any more faithful.

The catch is that **temperature is no longer a free parameter on current models**:

| Model | `temperature=1.0` | any other value |
|---|---|---|
| `claude-sonnet-5` | accepted (it is the default) | **400** |
| `claude-opus-5`, `fable-5`, `opus-4.8`, `opus-4.7` | **400** - parameter removed | **400** |

Since **1.0 is the API default**, passing it and omitting it are the same request. `sampling_args()`
just picks whichever form the configured model accepts, so changing `LLM_MODEL` in `config.py`
cannot produce a 400.

In [14]:
for m in ["claude-sonnet-5", "claude-opus-5", "claude-haiku-4-5"]:
    args = sampling_args(m)
    how = "passes temperature=1.0" if args else "omits it (same behaviour - 1.0 IS the default)"
    print(f"  {m:<20} -> {str(args):<22} {how}")

  claude-sonnet-5      -> {'temperature': 1.0}   passes temperature=1.0
  claude-opus-5        -> {}                     omits it (same behaviour - 1.0 IS the default)
  claude-haiku-4-5     -> {'temperature': 1.0}   passes temperature=1.0


### 6.2 Path 1 - the four-school panel (almost no LLM)

This is the most trustworthy part of the demo, and the reason is worth stating plainly:
**no model writes the text.** FiqhQA already contains each school's position as prose, so the card
body is copied through verbatim from `Position.text`.

The only generated thing is a six-value enum per school, for colour-coding - and one batched call
covers all four. If the enum is ever wrong, the reader can see it, because the position text it was
derived from is sitting directly underneath it.

In [15]:
QUERY_4S = "Does touching a woman break wudu?"
r4 = idx.retrieve(QUERY_4S)
panel_doc = r4["panel"]
print(f"panel: {panel_doc.id}  cos={panel_doc.score:.3f}  {panel_doc.title[:60]}")
print(f"stated positions: {len(panel_doc.stated_positions)} of 4\n")

if RUN_LLM:
    t0 = time.perf_counter()
    panel_cards = school_cards(panel_doc, QUERY_4S)
    print(f"1 batched call, {time.perf_counter()-t0:.1f}s\n")
    for c in panel_cards:
        print(f"  {c.attribution:<16} [{c.verdict}]")
        print(f"      {c.reasoning[:110]}...")
else:
    panel_cards = []
    print("(skipped - no ANTHROPIC_API_KEY)")

panel: fiqhqa:21  cos=0.797  The Nullification of Wudu by Touching a Woman
stated positions: 4 of 4



1 batched call, 2.6s

  Hanafi school    [permissible]
      According to the Hanafi school, ablution is not nullified by touch....
  Shafi'i school   [impermissible]
      According to the Shafi'i school, touching nullifies ablution in every case....
  Maliki school    [depends]
      According to the Maliki school, ablution is nullified by touch if it is done with intention of pleasure or ple...
  Hanbali school   [depends]
      The well-known opinion from Ahmad's school is that touching women with desire nullifies ablution, while it doe...


### 6.3 Path 2 - one structured call per single-source hit

One call, one document. The prompt hands over exactly one fatwa and says it is the only permissible
source; there is no cross-document context in the call at all, so the model *cannot* blend two
sources even if asked to.

`Doc.llm_context()` truncates the body to `MAX_ANSWER_CHARS_FOR_LLM` (6000). The verification in the
next cell runs against the **full** answer, not the truncated one - a real quote from the tail of a
long fatwa would otherwise be flagged as fabricated.

The response is constrained by a Pydantic schema via `client.messages.parse()`, so there is no
"regex the JSON out of the prose" step and no retry loop for malformed output.

In [16]:
hit = r4["hits"][0]
print(f"{hit.id}  {hit.source_label} ({hit.orientation})")
print(f"answer: {len(hit.answer)} chars -> llm_context: {len(hit.llm_context())} chars\n")

if RUN_LLM:
    t0 = time.perf_counter()
    card = source_card(hit, QUERY_4S)
    print(f"1 call, {time.perf_counter()-t0:.1f}s\n")
    print(f"  {card.attribution}  [{card.verdict}]")
    print(f"  {card.one_line}")
    for cond in card.conditions:
        print(f"    * {cond}")
    for e in card.evidences:
        print(f"    [verified {e['type']}] {e['ref']}: \"{e['quote'][:70]}...\"")
    for q in card.unverified_quotes:
        print(f"    [UNVERIFIED] {q[:70]}...")
else:
    card = None
    print("(skipped - no ANTHROPIC_API_KEY)")

islamqaorg:131249  IslamQA.org (Hanafi)
answer: 193 chars -> llm_context: 193 chars



1 call, 12.2s

  IslamQA.org (Hanafi)  [permissible]
  According to the Hanafi Mazhab, touching the private part does not break wudu.
    [verified scholarly] Hanafi Mazhab / Mufti Zakaria Makada: "According to the Hanafi Mazhab it does not break the wudu...."


#### Citation verification, demonstrated

The single most likely failure mode of a fatwa RAG is a **fluent, plausible, non-existent hadith**.
Prompting alone does not fix it, so every quote is checked back against the source text before it
renders.

The check normalises punctuation and whitespace first, because a genuine quote and its source
routinely differ by a curly apostrophe or a line wrap - a strict substring test would flag those too,
and a guard that cries wolf gets ignored.

Failures are moved to `unverified_quotes`, **not deleted**. A silent drop would make the model look
like it behaved.

In [17]:
from data.raw.schema import RulingCard  # noqa: E402

probe = RulingCard(
    doc_id=hit.id, attribution="test", verdict="depends", one_line="", reasoning="",
    evidences=[
        {"type": "scholarly", "ref": "genuine", "quote": hit.answer[40:140]},
        {"type": "hadith", "ref": "Bukhari 9999", "quote":
         "The Prophet said: whoever touches a woman must renew his ablution seven times."},
    ],
)
verify_quotes(probe, f"{hit.title}\n{hit.question}\n{hit.answer}")
print(f"kept       {len(probe.evidences)}  <- copied out of the document")
print(f"flagged    {len(probe.unverified_quotes)}  <- fabricated for this demo")
for q in probe.unverified_quotes:
    print(f"           \"{q[:80]}...\"")

kept       1  <- copied out of the document
flagged    1  <- fabricated for this demo
           "The Prophet said: whoever touches a woman must renew his ablution seven times...."


### 6.4 The comparison layer - the part that must not adjudicate

One call over the finished cards. The prompt forbids "the correct view is" / "the stronger opinion",
but the real guarantee is structural: **`Comparison` has no field for an overall verdict**, so a
schema-constrained response has nowhere to put one.

`turns_on` is what makes the panel useful rather than merely plural - it names the single underlying
question the disagreement reduces to (a definition, a hadith's authenticity, the scope of a
condition). A reader should finish able to state each position accurately, not able to say who won.

`disambiguate()` runs first: `MAX_PER_SOURCE = 2` lets one site occupy two slots, and two fatwas
from the same site can genuinely differ. Without the suffix both cards read `IslamQA.info (Salafi)`
and the comparison appears to show a source contradicting itself.

In [18]:
if RUN_LLM:
    cards = disambiguate(panel_cards + [card])
    t0 = time.perf_counter()
    cmp_ = compare(cards, QUERY_4S)
    print(f"1 call over {len(cards)} cards, {time.perf_counter()-t0:.1f}s\n")
    for a in cmp_.agreement:
        print(f"  agree:   {a}")
    for d in cmp_.divergence:
        print(f"  diverge: {d['point']}")
        for p in d["positions"]:
            print(f"             {p['who']:<22} {p['stance'][:60]}")
    print(f"\n  turns on: {cmp_.turns_on}")
    print(f"\n  fields on Comparison: {list(cmp_.__dataclass_fields__)}  <- no verdict field")
else:
    print("(skipped - no ANTHROPIC_API_KEY)")

1 call over 5 cards, 14.0s

  agree:   Hanafi school and IslamQA.org (Hanafi) both hold that touching does not nullify ablution.
  diverge: Does touching a woman nullify wudu, and under what conditions?
             Hanafi school          Ablution is not nullified by touch, without qualification.
             Shafi'i school         Touching nullifies ablution in every case.
             Maliki school          Nullifies only if done with intention of pleasure or if plea
             Hanbali school         Nullifies if touching is with desire; does not nullify if to
             IslamQA.org (Hanafi)   States plainly that touching the private part does not nulli
  diverge: What kind of contact is being addressed by the ruling?
             Hanafi school          Addresses touching (of a woman) generally as not nullifying 
             IslamQA.org (Hanafi)   Specifically addresses touching the private part, not touchi

  turns on: The disagreement reduces to whether desire or pleasure is a

### 6.5 The whole pipeline, and the abstain path

`answer(query, idx)` is the single function `app.py` calls. It runs retrieval, picks the path, and
returns everything the UI renders.

The abstain case is the one to watch. **A RAG that knows when to shut up beats one that always
answers** - and this is not decoration, it is the difference between the system returning
"I have no sourced fatwa on this" and returning four confident school positions about *bathrooms*
in response to a question about prayer on Mars. That is literally what happened before the
thresholds were recalibrated in section 4.1.

Note `llm_calls=0` on the abstain path: the guard runs before any generation, so refusing is free.

In [19]:
if RUN_LLM:
    t0 = time.perf_counter()
    a_out = answer("What are the rules for prayer on Mars?", idx)
    print(render(a_out))
    print(f"\n({time.perf_counter()-t0:.1f}s)\n")

    t0 = time.perf_counter()
    a_out = answer("Where do you place your hands in prayer?", idx, max_source_cards=2)
    print(render(a_out))
    print(f"\n({time.perf_counter()-t0:.1f}s, {a_out.llm_calls} calls, "
          f"{len(a_out.unverified)} unverified quotes)")
else:
    print("(skipped - no ANTHROPIC_API_KEY)")

=== What are the rules for prayer on Mars?
    [abstain] top=0.710  llm_calls=0
    I don't have a sourced fatwa on this. Nothing in the corpus is close enough to the question to answer from, and answering anyway would mean inventing one.

(0.0s)



=== Where do you place your hands in prayer?
    [four_school] top=0.843  llm_calls=4

  -- Hanafi school  [recommended]  (fiqhqa:1)
     The Hanafis hold that placing the right hand over the left (qabd) is a Sunnah act in prayer.

  -- Shafi'i school  [recommended]  (fiqhqa:1)
     The Shafi’is hold that placing the right hand over the left (qabd) is a Sunnah act in prayer.

  -- Maliki school  [depends]  (fiqhqa:1)
     The Malikis, disagreed, stating that letting the hands rest naturally (irsal) is recommended, while qabd is disliked in obligatory prayers. However, they permitted it in voluntary prayers, as previous

  -- Hanbali school  [recommended]  (fiqhqa:1)
     The Hanbalis hold that placing the right hand over the left (qabd) is a Sunnah act in prayer.

  -- IslamQA.org (Hanbali) #154072  [recommended]  (islamqaorg:154072)
     It is preferred to grasp the left wrist with the right hand and place them under the navel.
     [scholarly] Sharh al-Muntaha: "It is preferred that 

### 6.6 What this costs

Per answered query, with `max_source_cards=3`:

| Path | Calls | Notes |
|---|---|---|
| four-school panel | 1 | enums only; the prose is copied, not generated |
| single-source cards | 3 | run **concurrently** - wall-clock is one call, not three |
| comparison | 1 | |
| **total** | **5** | ~15-25s wall-clock |
| abstain | **0** | the guard runs before generation |

Retrieval is ~40 ms of that. **Generation is >99% of the latency and 100% of the cost**, which is
the argument for the diversity cap and for `max_source_cards`: every extra card is a round trip, and
the fourth and fifth hits rarely add a position the first three did not already state.

Two knobs if it is too slow: drop `max_source_cards` to 2, or skip the comparison call when the
verdicts are unanimous (nothing to compare).

**One honest limitation.** `VERDICTS` is a permissibility vocabulary - permissible, impermissible,
disliked, recommended, obligatory, depends. It fits "is a mortgage halal?" and fits *validity*
questions badly: "does touching break wudu?" is not really a permissible/impermissible question, and
forcing it into that enum reads oddly on the card. The card body stays accurate because it is copied
from the source; only the colour-coding badge is strained. Fixing it properly means a second enum in
`schema.py` for validity rulings.